<a href="https://colab.research.google.com/github/samdvey/MKTG6203DigitalTwins/blob/main/InfluencerWealth_Phase2_RAG_Quantitative.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: RAG-Grounded Synthetic Interviews — Quantitative Likert Version
### Using Real Twin-2K-500 Participants as Digital Twins
**Professor Joseph Pancras | University of Connecticut School of Business**

---

## What this version does

This notebook keeps the same **RAG-based digital twin workflow** as the corrected qualitative version, but changes the output format to **quantitative Likert-style responses** rather than open-ended interview answers.

Each selected participant is asked to respond **in character** to a set of survey items about influencer wealth perception using a **1–7 Likert scale**:

- **1 = Strongly disagree**
- **2 = Disagree**
- **3 = Somewhat disagree**
- **4 = Neutral**
- **5 = Somewhat agree**
- **6 = Agree**
- **7 = Strongly agree**

This makes the output easier to:

- compare across participant profiles,
- export to CSV,
- visualize in charts,
- and use for simple quantitative analysis in a seminar paper or presentation.

---

## RAG logic stays the same

**RAG = Retrieval-Augmented Generation.** Instead of generating answers from scratch, the model first receives a participant's full psychological profile and then produces Likert ratings that are consistent with that real participant's measured traits.

So the **retrieval step** is unchanged. The main difference is that the final responses are now **structured, numeric, and analysis-ready**.


In [ ]:
# CELL 1: Install required packages
# This may take 1-2 minutes. Lots of text scrolling is normal.

!pip install openai datasets pandas --quiet
print("Packages installed. Ready for Cell 2.")

In [ ]:
# CELL 2: Enter your OpenAI API key
# When you run this cell, a box will appear. Paste your key there and press Enter.
# Your key will NOT be visible on screen.

import getpass
from openai import OpenAI

api_key = getpass.getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)
print("API key accepted.")


In [ ]:
# CELL 3: Load all 2,058 participants from Twin-2K-500
#
# WHY RAG NEEDS THE FULL DATASET:
# Phase 1 defined 5 personas by hand. Here we load all 2,058 real participants
# so we can SEARCH for the ones whose actual measured traits best match the
# psychological profiles most relevant to influencer wealth perception research.
# This is the "retrieval" step in RAG.
#
# Runtime: ~1-2 minutes. Watch for the progress bars.

import pandas as pd
from datasets import load_dataset

print("Loading Twin-2K-500 dataset from Hugging Face...")
print("(Download progress bars will appear — this is normal)")
print()

chunk_files = [
    "full_persona/chunks/persona_chunk_001.parquet",
    "full_persona/chunks/persona_chunk_002.parquet",
    "full_persona/chunks/persona_chunk_003.parquet",
    "full_persona/chunks/persona_chunk_004.parquet",
    "full_persona/chunks/persona_chunk_005.parquet",
    "full_persona/chunks/persona_chunk_006.parquet",
    "full_persona/chunks/persona_chunk_007.parquet",
]

all_dfs = []
for i, chunk_file in enumerate(chunk_files, 1):
    ds = load_dataset(
        "LLM-Digital-Twin/Twin-2K-500",
        data_files=chunk_file,
        split="train"
    )
    df_chunk = ds.to_pandas()
    all_dfs.append(df_chunk)
    print(f"  Chunk {i}/7 loaded: {len(df_chunk)} participants")

df = pd.concat(all_dfs, ignore_index=True)
print()
print(f"SUCCESS: Loaded {len(df):,} participants total")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ============================================================================
# REPLACE CELL 4 WITH THIS - Using Traits That Actually Exist
# ============================================================================

import re
import numpy as np

def extract_score(text, score_name):
    """Extracts numeric score from persona_summary text."""
    if pd.isna(text):
        return None
    pattern = rf"{score_name}\s*=\s*([\-0-9\.]+)"
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        try:
            return float(match.group(1))
        except:
            return None
    return None

def extract_percentile(text, score_name):
    """Extracts percentile rank from persona_summary text."""
    if pd.isna(text):
        return None
    pattern = rf"{score_name}\s*=\s*[\-0-9\.]+\s*\((\d+)(?:st|nd|rd|th) percentile\)"
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        try:
            return int(match.group(1))
        except:
            return None
    return None

print("Extracting psychological traits for influencer wealth research...")
print()
print("TRAIT SELECTION RATIONALE:")
print("-" * 70)
print("For studying influencer wealth perception, we use:")
print()
print("1. EXTRAVERSION (score_extraversion)")
print("   High = outgoing, socially engaged, status-aware")
print("   Low = reserved, less influenced by social trends")
print("   → Predicts engagement with influencer content")
print()
print("2. AGREEABLENESS (score_agreeableness)")
print("   High = trusting, conforming, follows social norms")
print("   Low = skeptical, critical, questions motives")
print("   → Predicts whether wealth displays are seen as aspirational or suspicious")
print()
print("3. MINIMALISM (score_minimalism)")
print("   High = anti-consumerism, values simplicity")
print("   Low = materialistic, values possessions/status")
print("   → Directly measures attitude toward conspicuous consumption")
print()
print("These three traits create meaningful segments:")
print("  - High E + Low A + Low M = Status aspirational (wants influencer lifestyle)")
print("  - Low E + High A + High M = Anti-materialist (rejects wealth displays)")
print("  - High E + Low A + High M = Conflicted critic (notices but disapproves)")
print("=" * 70)
print()

# Extract the three traits that EXIST in the dataset
df["extraversion"] = df["persona_summary"].apply(lambda x: extract_score(x, "score_extraversion"))
df["agreeableness"] = df["persona_summary"].apply(lambda x: extract_score(x, "score_agreeableness"))
df["minimalism"] = df["persona_summary"].apply(lambda x: extract_score(x, "score_minimalism"))

df["extra_pct"] = df["persona_summary"].apply(lambda x: extract_percentile(x, "score_extraversion"))
df["agree_pct"] = df["persona_summary"].apply(lambda x: extract_percentile(x, "score_agreeableness"))
df["min_pct"] = df["persona_summary"].apply(lambda x: extract_percentile(x, "score_minimalism"))

# Clean dataset
df_clean = df.dropna(subset=["extraversion", "agreeableness", "minimalism"]).copy()

print(f"Participants with complete scores: {len(df_clean):,} of {len(df):,}")
print()
print("Score ranges:")
print(f"  Extraversion:  min={df_clean['extraversion'].min():.2f}, max={df_clean['extraversion'].max():.2f}")
print(f"  Agreeableness: min={df_clean['agreeableness'].min():.2f}, max={df_clean['agreeableness'].max():.2f}")
print(f"  Minimalism:    min={df_clean['minimalism'].min():.2f}, max={df_clean['minimalism'].max():.2f}")
print()
print("✓ Ready to select participants!")

In [ ]:

# ============================================================================
# REPLACE CELL 5 WITH THIS - Participant Selection
# ============================================================================

print()
print("=" * 70)
print("SELECTING 5 PARTICIPANTS - PHASE 2 DIGITAL TWIN ARCHETYPES")
print("=" * 70)
print()

# Normalize scores to 0-1 range
for col in ["extraversion", "agreeableness", "minimalism"]:
    col_min = df_clean[col].min()
    col_max = df_clean[col].max()
    df_clean[f"{col}_norm"] = (df_clean[col] - col_min) / (col_max - col_min)

# Drop any NaN in normalized columns
df_clean = df_clean.dropna(subset=["extraversion_norm", "agreeableness_norm", "minimalism_norm"])

print(f"Pool of {len(df_clean):,} participants with complete trait profiles")
print()

# Define 5 archetype profiles for influencer wealth research
target_profiles = {
    "P1_Status_Aspirational": {
        "extra": 0.85,  # Very extraverted (socially engaged)
        "agree": 0.20,  # Low agreeableness (not easily influenced)
        "min": 0.15,    # Anti-minimalist (materialistic)
        "description": "Engaged with influencer culture, aspires to luxury lifestyle"
    },
    "P2_Anti_Materialist": {
        "extra": 0.15,  # Introverted (less social media engagement)
        "agree": 0.80,  # Highly agreeable (values community over status)
        "min": 0.85,    # Strongly minimalist
        "description": "Rejects conspicuous consumption, values simplicity"
    },
    "P3_Social_Moderate": {
        "extra": 0.50,  # Moderately extraverted
        "agree": 0.50,  # Moderately agreeable
        "min": 0.50,    # Moderate on materialism
        "description": "Balanced perspective, context-dependent reactions"
    },
    "P4_Critical_Observer": {
        "extra": 0.75,  # Extraverted (notices influencer content)
        "agree": 0.25,  # Low agreeableness (skeptical, critical)
        "min": 0.30,    # Somewhat materialistic but not extreme
        "description": "Aware of influencer culture but critical of wealth displays"
    },
    "P5_Conflicted_Minimalist": {
        "extra": 0.85,  # Very extraverted (highly engaged socially)
        "agree": 0.30,  # Low agreeableness (independent thinker)
        "min": 0.85,    # Strongly minimalist
        "description": "THEORETICALLY INTERESTING: Socially engaged but anti-materialist"
    }
}

selected_ids = []
selected_participants = []

for profile_name, targets in target_profiles.items():
    available = df_clean[~df_clean["pid"].isin(selected_ids)].copy()

    if len(available) == 0:
        print(f"⚠️  No participants available for {profile_name}")
        continue

    # Calculate Euclidean distance to target profile
    available["distance"] = np.sqrt(
        (available["extraversion_norm"].astype(float) - targets["extra"]) ** 2 +
        (available["agreeableness_norm"].astype(float) - targets["agree"]) ** 2 +
        (available["minimalism_norm"].astype(float) - targets["min"]) ** 2
    ).astype(float)

    # Safety check
    if available["distance"].dtype == 'object':
        available["distance"] = pd.to_numeric(available["distance"], errors='coerce')
        available = available.dropna(subset=["distance"])

    if len(available) == 0:
        print(f"⚠️  All participants eliminated for {profile_name}")
        continue

    # Select best match
    best = available.nsmallest(1, "distance").iloc[0]
    selected_ids.append(best["pid"])
    selected_participants.append({
        "profile": profile_name,
        "description": targets["description"],
        "pid": best["pid"],
        "extraversion": best["extraversion"],
        "extra_pct": best["extra_pct"],
        "agreeableness": best["agreeableness"],
        "agree_pct": best["agree_pct"],
        "minimalism": best["minimalism"],
        "min_pct": best["min_pct"],
        "distance": best["distance"],
        "persona_summary": best["persona_summary"],
    })
    print(f"✓ {profile_name}: PID {best['pid']} (match quality: {best['distance']:.3f})")

print()
print("=" * 70)
print(f"SUCCESSFULLY SELECTED {len(selected_participants)} PARTICIPANTS")
print("=" * 70)
print()

# Display selected participants
for p in selected_participants:
    print(f"PROFILE: {p['profile']}")
    print(f"  {p['description']}")
    print(f"  PID: {p['pid']}")
    print(f"  Extraversion:  {p['extraversion']:.2f} ({p['extra_pct']}th percentile)")
    print(f"  Agreeableness: {p['agreeableness']:.2f} ({p['agree_pct']}th percentile)")
    print(f"  Minimalism:    {p['minimalism']:.2f} ({p['min_pct']}th percentile)")
    print(f"  Match quality: {p['distance']:.3f} (0 = perfect match)")
    print("-" * 70)
    print()

print()
print("=" * 70)
print("THEORETICAL MAPPING TO INFLUENCER WEALTH PERCEPTION")
print("=" * 70)
print()
print("P1 (Status Aspirational): High extraversion + Low minimalism")
print("  → Likely to ADMIRE influencer wealth displays")
print("  → Views luxury as aspirational and desirable")
print()
print("P2 (Anti-Materialist): Low extraversion + High minimalism")
print("  → Likely to REJECT influencer wealth displays")
print("  → Values simplicity and authenticity over status")
print()
print("P3 (Social Moderate): Balanced on all traits")
print("  → Context-dependent reactions")
print("  → May appreciate tasteful displays, reject excess")
print()
print("P4 (Critical Observer): High extraversion + Low agreeableness")
print("  → AWARE of influencer content but SKEPTICAL")
print("  → May see wealth displays as inauthentic or manipulative")
print()
print("P5 (Conflicted Minimalist): High extraversion + High minimalism")
print("  → THE THEORETICALLY INTERESTING CASE")
print("  → Socially engaged BUT anti-materialist")
print("  → May feel tension between social norms and personal values")
print("  → Could show ambivalent or complex reactions")
print()
print("=" * 70)
print("These profiles replace the Phase 1 fictional personas:")
print("  P1 ≈ Zoe (aspirational)")
print("  P2 ≈ Marcus (critical/anti-materialist)")
print("  P3 ≈ Priya (moderate/balanced)")
print("  P4 ≈ Derek (analytical/skeptical)")
print("  P5 ≈ Aaliyah (conflicted critic)")
print("=" * 70)
print()
print("✓ Ready for Phase 2 RAG interviews!")

In [ ]:
# CELL 2: Enter your OpenAI API key
# When you run this cell, a box will appear. Paste your key there and press Enter.
# Your key will NOT be visible on screen.

import getpass
from openai import OpenAI

api_key = getpass.getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)
print("API key accepted.")


In [ ]:
# CELL 6: Run RAG-grounded quantitative Likert survey
#
# The model reads each participant's full persona summary, then returns
# numeric 1-7 Likert ratings as if it were that participant.
#
# Runtime: ~1-2 minutes.

import json
import pandas as pd

likert_scale = {
    1: "Strongly disagree",
    2: "Disagree",
    3: "Somewhat disagree",
    4: "Neutral",
    5: "Somewhat agree",
    6: "Agree",
    7: "Strongly agree"
}

survey_items = [
    {"id": "L1", "construct": "admiration", "text": "When influencers show luxury purchases or expensive travel, I usually see it as motivating or aspirational."},
    {"id": "L2", "construct": "skepticism", "text": "When influencers show wealth online, I become skeptical about how authentic they really are."},
    {"id": "L3", "construct": "purchase_intent", "text": "Seeing an influencer display wealth makes me more interested in buying the products they promote."},
    {"id": "L4", "construct": "fairness", "text": "It matters to me whether an influencer's wealth comes from hard work rather than family money or debt."},
    {"id": "L5", "construct": "materialism_rejection", "text": "Lavish displays of wealth on social media usually turn me off rather than impress me."},
    {"id": "L6", "construct": "social_comparison", "text": "Posts showing influencer wealth make me compare my own lifestyle to theirs."},
    {"id": "L7", "construct": "trust", "text": "I trust influencers less when they frequently flaunt expensive purchases."},
    {"id": "L8", "construct": "engagement", "text": "Even if I dislike wealth displays, I still pay attention to that kind of content."},
    {"id": "L9", "construct": "behavior_change", "text": "I would be willing to unfollow an influencer if their wealth display felt excessive or out of touch."},
    {"id": "L10", "construct": "normalization", "text": "Showing off wealth is just a normal part of influencer culture and does not bother me much."}
]

system_prompt_prefix = """You are a research participant in a quantitative study about social media influencers and wealth perception.
Your complete psychological profile, based on validated survey instruments, is provided below.
Respond as this real participant would respond, staying consistent with the profile.

You must answer using ONLY valid JSON.
Return one rating for every survey item using the 1-7 Likert scale below:
1 = Strongly disagree
2 = Disagree
3 = Somewhat disagree
4 = Neutral
5 = Somewhat agree
6 = Agree
7 = Strongly agree

Important rules:
- Stay psychologically consistent with the participant profile.
- Use the full profile, not stereotypes.
- Output integers only for ratings.
- Do not skip any item.
- Keep rationales brief.

Return JSON with exactly this structure:
{
  "ratings": [
    {"id": "L1", "rating": 1, "rationale": "brief explanation"}
  ]
}

YOUR COMPLETE PSYCHOLOGICAL PROFILE:
========================
"""

survey_text = "\n".join([
    f"{item['id']} ({item['construct']}): {item['text']}" for item in survey_items
])

print("Running Phase 2 RAG-grounded quantitative Likert survey...")
print("(GPT reads each participant's psychological profile before rating each item)")
print()

results = []

for i, participant in enumerate(selected_participants, 1):
    print(f"Scoring participant {i}/5: PID {participant['pid']} ({participant['profile']})...")

    user_message = f"Rate the following survey items as this participant.\n\n{survey_text}"
    system_message = system_prompt_prefix + participant["persona_summary"]

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=0.3,
        max_tokens=1400,
        response_format={"type": "json_object"}
    )

    raw_answer = response.choices[0].message.content
    parsed = json.loads(raw_answer)
    rating_rows = parsed["ratings"]

    row = {
        "profile": participant["profile"],
        "description": participant["description"],
        "pid": participant["pid"],
        "extra_pct": participant["extra_pct"],
        "agree_pct": participant["agree_pct"],
        "min_pct": participant["min_pct"]
    }

    for item in survey_items:
        match = next((r for r in rating_rows if r.get("id") == item["id"]), None)
        if match is None:
            raise ValueError(f"Missing rating for {item['id']} from participant {participant['pid']}")
        row[item["id"]] = int(match["rating"])
        row[f"{item['id']}_rationale"] = match.get("rationale", "")

    results.append(row)
    print("  Done.")

results_df = pd.DataFrame(results)

construct_map = {item["id"]: item["construct"] for item in survey_items}
construct_scores = pd.DataFrame(index=results_df.index)

for construct in sorted(set(construct_map.values())):
    construct_items = [item_id for item_id, c in construct_map.items() if c == construct]
    construct_scores[construct] = results_df[construct_items].mean(axis=1)

results_with_constructs = pd.concat([
    results_df[["profile", "description", "pid", "extra_pct", "agree_pct", "min_pct"]],
    construct_scores,
    results_df[[item["id"] for item in survey_items]]
], axis=1)

results_df.to_csv("phase2_likert_full_results.csv", index=False)
results_with_constructs.to_csv("phase2_likert_summary_results.csv", index=False)

print()
print("All 5 quantitative profiles complete.")
print("Saved files:")
print("- phase2_likert_full_results.csv")
print("- phase2_likert_summary_results.csv")
print("Run the next cell to inspect the results.")

In [ ]:
# CELL 7: Display quantitative Likert results

print("=" * 80)
print("PHASE 2 RAG-GROUNDED QUANTITATIVE LIKERT RESULTS")
print("Twin-2K-500 | Toubia et al. (2025)")
print("=" * 80)
print()

item_labels = {
    "L1": "Admiration / aspiration",
    "L2": "Skepticism about authenticity",
    "L3": "Purchase interest",
    "L4": "Importance of wealth source",
    "L5": "Rejection of conspicuous consumption",
    "L6": "Social comparison",
    "L7": "Reduced trust",
    "L8": "Continued attention / engagement",
    "L9": "Willingness to unfollow",
    "L10": "Normalization of wealth display"
}

print("LIKERT SCALE:")
for k, v in likert_scale.items():
    print(f"{k} = {v}")
print()

print("PARTICIPANT-LEVEL RESULTS")
print("-" * 80)
for _, r in results_with_constructs.iterrows():
    print()
    print(f"PID {r['pid']} | {r['profile']}")
    print(f"Description: {r['description']}")
    print(f"Traits: Extraversion={r['extra_pct']}th | Agreeableness={r['agree_pct']}th | Minimalism={r['min_pct']}th")
    print("Construct means:")
    construct_cols = [c for c in results_with_constructs.columns if c not in ['profile', 'description', 'pid', 'extra_pct', 'agree_pct', 'min_pct'] + list(item_labels.keys())]
    for c in construct_cols:
        print(f"  - {c}: {r[c]:.2f}")
    print("Item ratings:")
    for item_id, label in item_labels.items():
        score = int(r[item_id])
        print(f"  - {item_id}: {score} ({likert_scale[score]}) | {label}")
    print("-" * 80)

print()
print("SUMMARY TABLE")
print("-" * 80)
summary_cols = ["profile", "pid", "extra_pct", "agree_pct", "min_pct"] + [c for c in results_with_constructs.columns if c not in ['profile', 'description', 'pid', 'extra_pct', 'agree_pct', 'min_pct'] + list(item_labels.keys())]
print(results_with_constructs[summary_cols].round(2).to_string(index=False))

print()
print("ITEM-LEVEL TABLE")
print("-" * 80)
print(results_with_constructs[["profile", "pid"] + list(item_labels.keys())].to_string(index=False))

print()
print("=" * 80)
print("SUGGESTED NEXT STEPS")
print("=" * 80)
print("1. Compare high vs. low minimalism profiles on L1, L5, and L10.")
print("2. Compare low vs. high agreeableness profiles on L2, L4, and L7.")
print("3. Use the CSV files for bar charts, heatmaps, or profile comparisons.")
print("4. Interpret especially interesting mixed profiles, such as high extraversion + high minimalism.")
print("5. Optionally add more items and rerun the notebook for a larger synthetic instrument.")
print("=" * 80)
